### JSON Output Parser - Practice & Exercises

In this notebook, you will practice using `JsonOutputParser` to generate and parse structured JSON data as standard Python dictionaries.

#### Part 1: Follow-Along Practice

##### Method 1: JSON Output Parser without Schema (Freeform JSON)

In [ ]:
from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain.chat_models import init_chat_model

# Load environment variables
load_dotenv()

# Initialize Groq model
model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

# Instantiate freeform json parser
json_parser = JsonOutputParser()
format_instructions = json_parser.get_format_instructions()

# Create prompt
prompt_free = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant that outputs structured data in JSON. {format_instructions}"),
    ("human", "List the top 3 cities in France and include their population and a famous landmark.")
])

# Format, Invoke, and Parse
messages_free = prompt_free.format_messages(format_instructions=format_instructions)
response_free = model.invoke(messages_free)
result_free = json_parser.parse(response_free.content)

print(result_free)
print(f"Type: {type(result_free)}")

##### Method 2: JSON Output Parser with Schema

In [ ]:
from pydantic import BaseModel, Field

# Define target model using Pydantic
class MovieInfo(BaseModel):
    title: str = Field(description="The name of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")
    genres: list[str] = Field(description="Genres of the movie")

# Instantiate the parser with the schema
schema_parser = JsonOutputParser(pydantic_object=MovieInfo)
schema_instructions = schema_parser.get_format_instructions()

# Create prompt
prompt_schema = ChatPromptTemplate.from_messages([
    ("system", "Generate the details about the movie. {format_instructions}"),
    ("human", "Provide details for the movie 'Inception'.")
])

# Format, Invoke, and Parse
messages_schema = prompt_schema.format_messages(format_instructions=schema_instructions)
response_schema = model.invoke(messages_schema)
result_schema = schema_parser.parse(response_schema.content)

print(result_schema)
print(f"Type: {type(result_schema)}")

#### Part 2: Practice Exercises

##### Exercise: Car Info JSON Generator
Define a Pydantic class `CarInfo` containing fields for `make`, `model`, and `colors` (list of strings). Use `JsonOutputParser` to query and return a structured dictionary representing the car *Tesla Model S*.

In [ ]:
# TODO: Write your implementation here


<details>
<summary>💡 Click to see solution</summary>

```python
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

class CarInfo(BaseModel):
    make: str = Field(description="Brand/make of the car")
    model: str = Field(description="Model name of the car")
    colors: list[str] = Field(description="Available colors for the car")

parser = JsonOutputParser(pydantic_object=CarInfo)
instructions = parser.get_format_instructions()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Provide car details in JSON. {format_instructions}"),
    ("human", "Details for the Tesla Model S.")
])

messages = prompt.format_messages(format_instructions=instructions)
response = model.invoke(messages)
result = parser.parse(response.content)

print(result)
print(f"Make: {result.get('make')}")
```
</details>